-------------

Vamos a empezar de nuevo con Daowa Maad, misma estructura, mismo todo, pero le añadiré un canal extra para que vea en frecuencias bajas y ver si realiza buenas, malas o pésimas predicciones

-------------

In [ ]:
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
#                                           $
#  Importación de librerías                 $
#                                           $
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$

import os
import datetime
import cv2
import numpy as np
import torch
from typing import Any
from tqdm import tqdm
from torch import nn
from PIL import Image
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode, functional

In [3]:
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
#
# Función para probar solamente el código con un modelo mío
#
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$

from utils.models.utils_med import TransformerDaowa_maad

def get_DaowaV2(pretrained_weights:bool = False):
    if pretrained_weights:
        daowa = TransformerDaowa_maad(num_clases=4)
        daowa.load_state_dict(torch.load('../Models/Daowa_maadWeights.pth', weights_only=True))
    else:
        daowa = TransformerDaowa_maad(num_clases=4)
    
    return daowa


In [ ]:
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
#                                                            $
# Generación del dataset y aquí es donde ocurre la magia con $
# la descomposición de funciones de Fourier para imágenes    $
#                                                            $
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$

def get_fourier_lowpass(img_np, freq_radius):
    # Ya recibimos el array, no leemos disco otra vez
    h, w = img_np.shape
    f = np.fft.fft2(img_np)
    fshift = np.fft.fftshift(f)

    mask = np.zeros((h, w), np.float32)
    cy, cx = h // 2, w // 2
    cv2.circle(mask, (cx, cy), freq_radius, 1, -1)

    fshift_filtered = fshift * mask
    img_lowpass = np.abs(np.fft.ifft2(np.fft.ifftshift(fshift_filtered)))
    
    # Normalizamos a 0-1 para que coincida con ToTensor()
    return img_lowpass / 255.0

class CusDataset(Dataset):
    def __init__(self, dataframe, images_dir, masks_dir, images_transform=None, mask_transform = None, shape_img = tuple):
        self.df = dataframe #This is a Dataframe from Pandas
        self.img_dir = images_dir
        self.masks_dir = masks_dir
        self.shape = shape_img
        if images_transform:
            self.img_trans = images_transform
        else:
            self.img_trans = transforms.Compose([
                transforms.Resize((self.shape), interpolation=InterpolationMode.BILINEAR),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean = [0.4811, 0.4491, 0.3961],
                    std = [0.2634, 0.2587, 0.2667]
                )
            ])
        self.mask_transform = mask_transform
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, index):
        img_filename = self.df.iloc[index, 0]
        mask_filename = self.df.iloc[index, 1]

        img_dir = os.path.join(self.img_dir, img_filename)
        mask_dir = os.path.join(self.masks_dir, mask_filename)

        img = Image.open(img_dir).convert(mode='RGB').resize(self.shape)
        img_fourier_np = np.array(img.convert('L'))

        img_lowpass = get_fourier_lowpass(img_fourier_np, 50)
        img_tensor_lowpass = torch.from_numpy(img_lowpass).unsqueeze(0).float()
        
        
        mask = Image.open(mask_dir)

        img = self.img_trans(img)
        mask = functional.resize(mask, (self.shape), InterpolationMode.NEAREST)

        mask_np = np.array(mask)
        mask_tensor = torch.from_numpy(mask_np).long()
        mask_tensor = torch.clamp(mask_tensor, 0, 2)

        img = torch.cat((img, img_tensor_lowpass))
        return img, mask_tensor

In [ ]:
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
#                                                                            $
# Función de predicción                                                      $
#                                                                            $
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$




In [ ]:
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
#                                                             $
#   Codificación de la función de pérdida GeneralizedDiceLoss $
#                                                             $
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$

class GeneralizedDiceLossFN(nn.Module):
    def __init__(self, epsilon, target_classes):
        super(GeneralizedDiceLossFN, self).__init__()
        self.epsilon:float = epsilon
        self.target_classes:int = target_classes

    def forward(self, inTensor:torch.Tensor, target:torch.Tensor):

        """
        Attributes:
            - Input: Es el vector de entrada donde contiene los logits de la predicción, con shape = [Batch, N_clases, H, W]
            - Target: Es el tensor objetivo, donde se tiene que acercar, con shape = [Batch, H, W]
        """
        input = F.softmax(input=inTensor, dim = 1)
        input = input.view(input.size(0), input.size(1), -1).float()

        target = F.one_hot(target, self.target_classes).permute(0, 3, 1, 2).float()

        volumenes = torch.sum(target, dim = (2, 3))
        w_c = 1 / (volumenes ** 2 + self.epsilon)

        target = target.view(target.size(0), target.size(1), -1).float()

        interseccion = torch.sum(input * target, dim = 2)
        union = torch.sum(input + target, dim = 2)

        numerador = torch.sum(w_c * interseccion, dim = 1)
        denominador = torch.sum(w_c * union, dim = 1)

        out = 1 - (2 * (numerador / (denominador + self.epsilon)))
        
        return out.mean()



In [ ]:
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
#                                                              &
#  Codificación de la métrica de precisión IoU                 $
#                                                              &
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$


def IoU_global(intersection:torch.Tensor, union:torch.Tensor, smooth = 1e-6):
    
    IoU_por_clase:torch.Tensor = (intersection + smooth) / (union + smooth)
    
    IoU_global:torch.Tensor = torch.mean(IoU_por_clase)

    return IoU_global, IoU_por_clase

def meanIoU(y_pred:torch.Tensor, y_true:torch.Tensor, num_classes:int, smooth:float = 1e-6):

    y_pred = torch.argmax(y_pred, dim = 1)

    IoU_clase:list[Any] = []

    for i in range(0, num_classes):
        pred_i = (y_pred == i).to(torch.float32)
        true_i = (y_true == i).to(torch.float32)

        interseccion = torch.sum(pred_i * true_i)
        union = torch.sum(pred_i) + torch.sum(true_i) - interseccion #PRINCIPIO DE INCLUSIÓN-EXCLUSIÓN

        iou = (interseccion + smooth) / (union + smooth)
        IoU_clase.append(iou)
    
    if len(IoU_clase) == 0:
        return torch.tensor(0.0, device=y_pred.device)
    
    return torch.mean(torch.stack(IoU_clase))

def get_intersections_and_unions(y_pred:torch.Tensor, y_true:torch.Tensor, class_id:int | Any):

    y_pred = (y_pred == class_id).to(torch.float32)
    y_true = (y_true == class_id).to(torch.float32)

    intersection = torch.sum(y_pred * y_true)
    union = torch.sum(y_pred) + torch.sum(y_true) - intersection

    return  (intersection, union)

In [ ]:
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
#                                                                                $
#   Función de entrenamiento del modelo                                          $
#                                                                                $
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$

# Entrenamiento con Bfloat16 (Prueba)

def train_model(modelo, loss_fn, optimizador, dataloaders:list, device_calc):

    dia = datetime.date.today()
    history = {}

    if modelo and isinstance(dataloaders, list):
        
        modelo.to(device_calc)

        #Desempaquetado

        cross_fn, dice_fn = loss_fn

        train_dl, val_dl = dataloaders
        
        #Métricas
        epochs = 20
        best_iou = 0.0

        print("Iniciando entrenamiento")

        for i in tqdm(range(1, epochs + 1), desc=f"Entrenando el modelo"):
            if 0 < i <= 5:
                #Métricas
                peso_dice = 0
                peso_cross = 1
            
            elif 5 < i <= 10: 
                #Métricas
                peso_dice = 0.5
                peso_cross = 0.5
            
            elif 10 < i <= 20:
                #Métricas
                peso_dice = 0.8
                peso_cross = 0.2
            
            #Métricas
            train_loss_acc = 0.0
            train_correct_pixels = 0
            train_total_pixels = 0
            intersections = torch.zeros(3, device=device_calc)
            unions = torch.zeros(3, device=device_calc)

            #Modelo en modo de entrenamiento wachin
            modelo.train()
            for image, mask in train_dl:
                image, mask = image.to(device_calc), mask.to(device_calc)

                optimizador.zero_grad()
                with torch.amp.autocast_mode.autocast(device_type="cuda", dtype=torch.bfloat16):
                    output = modelo(image) # [Batch, 3, 512, 512]
                
                    loss_cross = cross_fn(output, mask)
                    loss_dice = dice_fn(output, mask) # Mask debe ser [Batch, 512, 512] (Long)
                    total_loss = (peso_cross * loss_cross + peso_dice * loss_dice)
                
                total_loss.backward()
                optimizador.step()

                train_loss_acc += total_loss.item()
                
                # Precisión de Píxeles (Train)
                _, preds = torch.max(output, 1) # Obtenemos el índice de la clase ganadora
                train_correct_pixels += (preds == mask).sum().item()
                train_total_pixels += mask.numel() # Total de píxeles en el batch

            train_loss = train_loss_acc / len(train_dl)
            train_acc = (train_correct_pixels / train_total_pixels) * 100

            #Ya nomás pa' evaluar
            actual_valLoss = 0.0
            val_correct_pixels = 0
            val_total_pixels = 0

            modelo.eval()
            with torch.no_grad():
                for image, mask in val_dl:
                    image = image.to(device_calc)
                    mask = mask.to(device_calc)
                    
                    #Predicción
                    predict = modelo(image)
                
                    _y_predicts = torch.argmax(predict, dim = 1)

                    inter_0, union_0 = get_intersections_and_unions(_y_predicts, mask, 0)
                    inter_1, union_1 = get_intersections_and_unions(_y_predicts, mask, 1)
                    inter_2, union_2 = get_intersections_and_unions(_y_predicts, mask, 2)

                    inters = torch.stack([inter_0, inter_1, inter_2])
                    unis = torch.stack([union_0, union_1, union_2])

                    intersections += inters
                    unions += unis

                    cross_loss = cross_fn(predict, mask)
                    dice_loss = dice_fn(predict, mask)
                    val_loss = (peso_cross * cross_loss + peso_dice * dice_loss)

                    #Acumular loss
                    actual_valLoss += val_loss.item()

                    _, predicts = torch.max(predict, 1) #Acá podes ver la confianza de predicción con confianza en lugar de _

                    val_correct_pixels += (predicts == mask).sum().item()
                    val_total_pixels += mask.numel()

            avg_val_loss = actual_valLoss / len(val_dl)
            val_acc = (val_correct_pixels / val_total_pixels) * 100
            mIoU, IoU_por_clase = IoU_global(intersections, unions)
            IoU_por_clase = IoU_por_clase.cpu().numpy().tolist()

            if scheduler:
                if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                    scheduler.step(avg_val_loss)
                else:
                    scheduler.step()
            
            if mIoU > best_iou:
                torch.save(modelo.state_dict(), fr'Models/ModeloPrueba{dia}.pth')
                best_iou = mIoU
                print(f"--> Nuevo mejor modelo con acc: {best_iou:.2f}")

            history[i] = {
                "train_loss":train_loss,
                "train_acc":train_acc, 
                "val_loss":avg_val_loss,
                "val_acc":val_acc,
                "val_iou Global":mIoU,
                "val_iou Clases":IoU_por_clase
                }
            
            print(f"Epoch {i}: Train Loss = {train_loss:.4f}; Precision = {train_acc:.4f}; Validation loss = {avg_val_loss:.4f}, Precisión = {val_acc:.4f}%, IoU Global = {mIoU:.4f}, IoU Clase [Mascota, Fondo, Borde] = {IoU_por_clase}")

2026-03-01


In [8]:
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
#                                                              &
#  Codificación de Attention Gates para filtrar contexto local $
#                                                              &
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$

class AttentionGates(nn.Module):
    def __init__(self, f_g, f_l, F_int):
        super(AttentionGates, self).__init__()

        self.w_x = nn.Sequential(
            nn.Conv2d(f_l, F_int, kernel_size= 3, stride = 2, padding = 1, bias = False),
            nn.BatchNorm2d(F_int, affine=True)
        )

        self.w_g = nn.Sequential(
            nn.Conv2d(f_g, F_int, kernel_size=3, stride=2, padding = 1, bias = False),
            nn.BatchNorm2d(F_int, affine = True)
        )
        
        self.psi = nn.Sequential(
            nn.Conv2d(in_channels = F_int, out_channels=1, kernel_size=1, stride = 1, padding = 0, bias = False),
            nn.BatchNorm2d(1, affine = tri)
        )

        self.relu = nn.ReLU(inplace=True)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, gating, skip):
        g = self.w_g(gating)
        s = self.w_x(skip)

        if g.shape[2:] != s.shape[2:]:
            g = torch.nn.functional.interpolate(g, size = s.shape[2:], mode='NEAREST')

        relu = self.relu(g + s)
        psi = self.psi(relu)
        sigmoid = self.sigmoid(psi)
        out = torch.nn.functional.interpolate(sigmoid, size = s.shape[2:], mode = 'bilinear', align_corners=True)
        
        return out

In [9]:

#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
#                                                              &
#  Codificación de un Transformer encoder para contexto global $
#                                                              &
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$


class EncoderTrans(nn.Module):
    def __init__(self, in_embed_dim, _num_heads, _bias, _batch_first):

        """
        Attributes:
            - in_embed_dim: Dimensiones de entrada, es decir, los canales de la imagen en el bottleneck
            - _num_heads: Número de cabezas de atención múltiple
            - _bias: Booleano para saber si se aplica o no un sesgo al módulo MultiheadAttention
            - _batch_first: Booleano para decirle al módulo: 'Oye, el primer elemento del tensor es el batch'
        """

        super(EncoderTrans, self).__init__()
        self.mha = nn.MultiheadAttention(
                in_embed_dim,
                _num_heads, 
                dropout = 0.2, 
                bias = _bias, 
                batch_first=_batch_first
            )
        self.normalization_layer = nn.RMSNorm(in_embed_dim)
        self.ffn= nn.Sequential(
            nn.Linear(in_features=in_embed_dim, out_features=in_embed_dim * 4),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(in_features=in_embed_dim * 4, out_features=in_embed_dim),
            nn.Dropout(0.2)
        )
    
    def forward(self, x):
        norm_x = self.normalization_layer(x)
        out_mha, _ = self.mha(norm_x, norm_x, norm_x)
        x = out_mha + x

        norm_x = self.normalization_layer(x)
        out_ffn = self.ffn(norm_x)
        x = x + out_ffn

        return x


In [10]:
for i in tqdm(range(10), desc="Entrenando a Daowa_maad", ascii=True):
    # Simulación de trabajo
    print(f"Epoch {i}: x")

Entrenando a Daowa_maad: 100%|##########| 10/10 [00:00<00:00, 102550.22it/s]

Epoch 0: x
Epoch 1: x
Epoch 2: x
Epoch 3: x
Epoch 4: x
Epoch 5: x
Epoch 6: x
Epoch 7: x
Epoch 8: x
Epoch 9: x


In [11]:
import pandas as pd

df = pd.read_csv('labels/dataset_daowaV2.csv')
ds = CusDataset(df, 'data/image/images/', 'data/image/masks/', None, None, (192, 192))
dl = DataLoader(ds, batch_size = 64, shuffle = False, pin_memory=True)

image, mask = next(iter(dl))
print(image.shape)

image = torch.argmax(image, dim = 1)

print(image.shape)
print(mask.shape)

torch.Size([64, 4, 192, 192])
torch.Size([64, 192, 192])
torch.Size([64, 192, 192])


In [ ]:
from typing import Literal

Intersecciones: list[float]= [0.5, 0.4, 0.2]
Uniones: list[float] = [0.4, 0.7, 0.6]

Interseccion_global: float | Literal[0] = sum(Intersecciones)
Union_global: float | Literal[0] = sum(Uniones)

IoU_global = Interseccion_global / Union_global

IoU_global

0.6470588235294118

In [33]:
hola = torch.zeros(3, device='cuda').cpu().numpy().tolist()
hola

[0.0, 0.0, 0.0]